### Importing Required Libraries

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries successfully imported")

Libraries successfully imported


- To display maximum rows and columns for the dataset

In [37]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",100)

print("Maximum display option is chosen")

Maximum display option is chosen


### Loading the Dataset

In [45]:
from pathlib import Path

print('Nb working dir:')
print(Path.cwd())

print("\nLookingforCSV:")
matches = list(Path.cwd().parent.rglob("HHS_Unaccompanied_Alien_Children_Program.csv"))

for path in matches:
    print(path.resolve())

Nb working dir:
c:\Users\ASUS\Int_Proj1\HHS_MedcareProj\notebooks

LookingforCSV:
C:\Users\ASUS\Int_Proj1\HHS_MedcareProj\notebooks\HHS_Unaccompanied_Alien_Children_Program.csv


In [44]:
df = pd.read_csv("HHS_Unaccompanied_Alien_Children_Program.csv", engine="python")

- Basic Description

In [7]:
print(df.shape)
print(df.columns.tolist())
display(df.head())
display(df.tail())

(720, 6)
['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6,18,11,"2,484",14
1,"December 18, 2025",11,50,6,"2,472",16
2,"December 17, 2025",7,31,11,"2,481",10
3,"December 16, 2025",8,54,15,"2,468",9
4,"December 15, 2025",11,42,9,"2,470",7


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
715,"January 25, 2023",20,22,41,"7,538",180
716,"January 24, 2023",47,42,47,"7,433",175
717,"January 23, 2023",32,50,39,"7,280",181
718,"January 22, 2023",32,49,39,"7,122",227
719,"January 12, 2023",33,53,34,"6,566",436


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype
---  ------                                           --------------  -----
 0   Date                                             720 non-null    str  
 1   Children apprehended and placed in CBP custody*  720 non-null    int64
 2   Children in CBP custody                          720 non-null    int64
 3   Children transferred out of CBP custody          720 non-null    int64
 4   Children in HHS Care                             720 non-null    str  
 5   Children discharged from HHS Care                720 non-null    int64
dtypes: int64(4), str(2)
memory usage: 48.1 KB


- Statistical Description

In [9]:
df.describe(include='all')

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
count,720,720.000000,720.000000,720.000000,720,720.000000
unique,720,NaN,NaN,NaN,642,NaN
top,"December 21, 2025",NaN,NaN,NaN,"2,340",NaN
freq,1,NaN,NaN,NaN,3,NaN
mean,NaN,93.523611,171.494444,128.668056,NaN,173.406944
std,NaN,72.646625,126.354965,97.322012,NaN,125.702841
min,NaN,0.000000,7.000000,0.000000,NaN,0.000000
25%,NaN,12.000000,36.000000,14.000000,NaN,19.750000
50%,NaN,99.000000,193.000000,157.000000,NaN,181.000000
75%,NaN,147.250000,263.250000,199.250000,NaN,267.000000


### Assessing Missing Values

In [10]:
missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().mean()*100
).round(2)

missing_summary = pd.DataFrame(
    {
        'Missing Values': missing_values,
        'Missing Percentage': missing_percentage
    }
)

missing_summary

,Missing Values,Missing Percentage
Date,0,0.0
Children apprehended and placed in CBP custody*,0,0.0
Children in CBP custody,0,0.0
Children transferred out of CBP custody,0,0.0
Children in HHS Care,0,0.0
Children discharged from HHS Care,0,0.0


### Duplicated Records

In [11]:
print('Duplicated Records:',df['Date'].duplicated().sum())

Duplicated Records: 0


### Convert Date

In [12]:
df['Date'] = pd.to_datetime(
    df['Date'],
    errors='coerce'
)

df = df.sort_values('Date').reset_index(drop=True)

df[['Date']].head()

print('Invalid Dates:', df['Date'].isna().sum())
print('Start Date:', df['Date'].min())
print('End Date:',df['Date'].max())

Invalid Dates: 0
Start Date: 2023-01-12 00:00:00
End Date: 2025-12-21 00:00:00


### Renaming Columns
- shortened and precised column names are always more helpful

In [13]:
df = df.rename(columns={"Children apprehended and placed in CBP custody*":"CBP_Intake",
                        "Children in CBP custody":"CBP_Custody",
                        "Children transferred out of CBP custody":"HHS_Transfers",
                        "Children in HHS Care":"HHS_Care",
                        "Children discharged from HHS Care":"HHS_Discharges"})

df.columns

Index(['Date', 'CBP_Intake', 'CBP_Custody', 'HHS_Transfers', 'HHS_Care',
       'HHS_Discharges'],
      dtype='str')

### Conversion into numeric dtypes

In [14]:
numeric_columns = [
    "CBP_Intake",
    "CBP_Custody",
    "HHS_Transfers",
    "HHS_Care",
    "HHS_Discharges"
]

for column in numeric_columns:
    df[column] = (
        df[column].astype(str).str.replace(",","",regex=False)
    )

    df[column] = pd.to_numeric(df[column],errors="coerce")

#checking
df.dtypes

Date              datetime64[us]
CBP_Intake                 int64
CBP_Custody                int64
HHS_Transfers              int64
HHS_Care                   int64
HHS_Discharges             int64
dtype: object

In [15]:
#Checking for missing values again

df[numeric_columns].isnull().sum()

CBP_Intake        0
CBP_Custody       0
HHS_Transfers     0
HHS_Care          0
HHS_Discharges    0
dtype: int64

### Reporting Date Analysis
- Missing dates will be treated as missing observations to prevent any false predictions
- Rather predictions will be adjusted accordingly

In [16]:
start_date = df['Date'].min()
end_date = df['Date'].max()

expected_dates = pd.date_range(
    start = start_date,
    end = end_date,
    freq='D'
)

observed_dates = pd.DatetimeIndex(df['Date'].dropna().unique())

missing_dates = expected_dates.difference(observed_dates)

print('First Reproting Date:',start_date.date())          #'.date()' method to get only the date and not the timing
print('Last Reproting Date:',end_date.date())
print('Expected Calendar Days:',len(expected_dates))
print('Observed Reporting Days:',len(observed_dates))
print('Missing Calendar Dates:',len(missing_dates))

First Reproting Date: 2023-01-12
Last Reproting Date: 2025-12-21
Expected Calendar Days: 1075
Observed Reporting Days: 720
Missing Calendar Dates: 355


### Reporting Gaps

In [17]:
df['Days_Since_Previous_Report'] = (df['Date'].diff().dt.days)
df['Days_Since_Previous_Report'].value_counts().sort_index()


Days_Since_Previous_Report
1.0     558
2.0      10
3.0     122
4.0      23
5.0       3
6.0       1
7.0       1
10.0      1
Name: count, dtype: int64

In [18]:
reporting_gaps = df[
    df['Days_Since_Previous_Report']>1
][
    ['Date','Days_Since_Previous_Report']
]

reporting_gaps.head(20)

,Date,Days_Since_Previous_Report
1,2023-01-22,10.0
5,2023-01-29,4.0
10,2023-02-05,3.0
15,2023-02-12,3.0
20,2023-02-20,4.0
24,2023-02-26,3.0
29,2023-03-07,5.0
32,2023-03-13,4.0
36,2023-03-19,3.0
41,2023-03-26,3.0


In [19]:
#Largest Gaps

reporting_gaps.sort_values(
    'Days_Since_Previous_Report',
    ascending=False
).head(10)

,Date,Days_Since_Previous_Report
1,2023-01-22,10.0
593,2025-06-17,7.0
138,2023-08-16,6.0
394,2024-08-27,5.0
29,2023-03-07,5.0
582,2025-05-27,5.0
86,2023-05-29,4.0
32,2023-03-13,4.0
5,2023-01-29,4.0
174,2023-10-09,4.0


### Logical Consistency

#### The following conditions are examined:
- HHS_Transfers don't exceed CBP_Custody in the reports
- HHS_Discharges don't exceed HHS_Care in the reports

#### If these conditions are found to be unfulfilled:
- Then they can be treated as proves that reported timing/dates might not be regularly updated

In [20]:
df['Transfer_Exceeds_CBP'] = (df['HHS_Transfers']>df['CBP_Custody'])
df['Discharge_Exceeds_HHS'] = (df['HHS_Discharges']>df['HHS_Care'])

print('Transfer > CBP Custody:', df['Transfer_Exceeds_CBP'].sum())
print('Discharge > HHS Care :', df['Discharge_Exceeds_HHS'].sum())

Transfer > CBP Custody: 86
Discharge > HHS Care : 0


#### Inspecting the Anomalies

In [21]:
anomalies = df[df['Transfer_Exceeds_CBP']]

anomalies[
    [
        'Date',
        'CBP_Custody',
        'HHS_Transfers',
        'HHS_Care',
        'HHS_Discharges'
    ]
].head(20)

,Date,CBP_Custody,HHS_Transfers,HHS_Care,HHS_Discharges
3,2023-01-24,42,47,7433,175
4,2023-01-25,22,41,7538,180
9,2023-02-02,13,23,7879,298
22,2023-02-22,215,230,7978,232
23,2023-02-23,162,178,7914,386
27,2023-03-01,158,193,7930,307
47,2023-04-03,139,168,8054,284
63,2023-04-25,230,242,8017,202
70,2023-05-04,253,286,8790,332
79,2023-05-17,162,193,8569,307


### Deriving Metrics

#### Total System Load
- Total no. of children being reported within the two care stages(CBP_Custody+HHS_Care)

In [22]:
df['Total_System_Load'] = (df['CBP_Custody']+df['HHS_Care'])

df[
    [
        'Date',
        'CBP_Custody',
        'HHS_Care',
        'Total_System_Load'
    ]
].head()

,Date,CBP_Custody,HHS_Care,Total_System_Load
0,2023-01-12,53,6566,6619
1,2023-01-22,49,7122,7171
2,2023-01-23,50,7280,7330
3,2023-01-24,42,7433,7475
4,2023-01-25,22,7538,7560


#### Net Intake
- Children leaving HHS Care through 'Transfers' and 'Discharges'

- Positive value: i.e. more transfers than discharges i.e. increased workload

In [23]:
df['Net_Intake'] = (df['HHS_Transfers']-df['HHS_Discharges'])

df[
    [
        'Date',
        'HHS_Transfers',
        'HHS_Discharges',
        'Net_Intake'
    ]
].head()

,Date,HHS_Transfers,HHS_Discharges,Net_Intake
0,2023-01-12,34,436,-402
1,2023-01-22,39,227,-188
2,2023-01-23,39,181,-142
3,2023-01-24,47,175,-128
4,2023-01-25,41,180,-139


#### Growth Rate

In [24]:
df['Care_Load_Growth_Rate'] = (df['Total_System_Load'].pct_change()*100)
df['Care_Load_Growth_Rate']

0           NaN
1      8.339628
2      2.217264
3      1.978172
4      1.137124
         ...   
715    0.600721
716    0.398089
717   -0.396511
718    0.398089
719   -0.793021
Name: Care_Load_Growth_Rate, Length: 720, dtype: float64

#### Rolling Metrics

In [25]:
df['Rolling_7D_Load'] = (
    df['Total_System_Load']
    .rolling(
        window=7,
        min_periods=3
    )
    .mean()
)

df['Rolling_14D_Load'] = (
    df['Total_System_Load']
    .rolling(
        window=14,
        min_periods=7
    )
    .mean()
)

print(df['Rolling_7D_Load'])
print(df['Rolling_14D_Load'])

0              NaN
1              NaN
2      7040.000000
3      7148.750000
4      7231.000000
          ...     
715    2482.714286
716    2492.571429
717    2499.000000
718    2506.000000
719    2507.285714
Name: Rolling_7D_Load, Length: 720, dtype: float64
0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
          ...     
715    2459.428571
716    2466.571429
717    2473.642857
718    2480.000000
719    2484.714286
Name: Rolling_14D_Load, Length: 720, dtype: float64


#### Cumulative Pressure

In [26]:
df['Cumulative_Net_Intake']= (df['Net_Intake'].cumsum())
df['Positive_Net_Intake']= (df['Net_Intake']>0)

print(df['Cumulative_Net_Intake'])
print(df['Positive_Net_Intake'])

0       -402
1       -590
2       -732
3       -860
4       -999
       ...  
715   -32206
716   -32200
717   -32199
718   -32209
719   -32212
Name: Cumulative_Net_Intake, Length: 720, dtype: int64
0      False
1      False
2      False
3      False
4      False
       ...  
715     True
716     True
717     True
718    False
719    False
Name: Positive_Net_Intake, Length: 720, dtype: bool


#### Volatility

In [27]:
df['Load_Volatility_7_Obs'] = (
    df['Total_System_Load']
    .rolling(
        window=7,
        min_periods=3
    )
    .std()
)

df['Load_Volatility_7_Obs']

0             NaN
1             NaN
2      373.163503
3      374.353109
4      372.733819
          ...    
715     19.972600
716     19.890175
717     17.358955
718     14.821156
719     13.864995
Name: Load_Volatility_7_Obs, Length: 720, dtype: float64

#### Discharge Offset Ratio

- =1.0 : Discharges equal to Transfers
- <1.0 : Transfers exceed Discharges
- >1.0 : Discharges exceed Transfers

In [28]:
df['Discharge_Offset_Ratio'] = np.where(df['HHS_Transfers']>0,df['HHS_Discharges']/df['HHS_Transfers'],np.nan)
df['Discharge_Offset_Ratio']

0      12.823529
1       5.820513
2       4.641026
3       3.723404
4       4.390244
         ...    
715     0.777778
716     0.600000
717     0.909091
718     2.666667
719     1.272727
Name: Discharge_Offset_Ratio, Length: 720, dtype: float64

#### Anomaly Flag

In [29]:
df['Anomaly_Flag'] = np.select(
    [
        df['Transfer_Exceeds_CBP'],
        df['Discharge_Exceeds_HHS']
    ],
    [
        'Transfer exceedds reported CBP custody',
        'Discharge exceeds reported HHS care'
    ],
    default='None'
)

df['Anomaly_Flag'].value_counts()

Anomaly_Flag
None                                      634
Transfer exceedds reported CBP custody     86
Name: count, dtype: int64

### Cleaned Dataset 

- Understanding the cleaned dataset

In [30]:
df.head()

print('Cleaned Dataset Shape:', df.shape)

print('\nCleaned Columns:')
print(df.columns.tolist())

df.describe()

Cleaned Dataset Shape: (720, 19)

Cleaned Columns:
['Date', 'CBP_Intake', 'CBP_Custody', 'HHS_Transfers', 'HHS_Care', 'HHS_Discharges', 'Days_Since_Previous_Report', 'Transfer_Exceeds_CBP', 'Discharge_Exceeds_HHS', 'Total_System_Load', 'Net_Intake', 'Care_Load_Growth_Rate', 'Rolling_7D_Load', 'Rolling_14D_Load', 'Cumulative_Net_Intake', 'Positive_Net_Intake', 'Load_Volatility_7_Obs', 'Discharge_Offset_Ratio', 'Anomaly_Flag']


,Date,CBP_Intake,CBP_Custody,HHS_Transfers,HHS_Care,HHS_Discharges,Days_Since_Previous_Report,Total_System_Load,Net_Intake,Care_Load_Growth_Rate,Rolling_7D_Load,Rolling_14D_Load,Cumulative_Net_Intake,Load_Volatility_7_Obs,Discharge_Offset_Ratio
count,720,720.000000,720.000000,720.000000,720.000000,720.000000,719.000000,720.000000,720.000000,719.000000,718.000000,714.000000,720.000000,718.000000,717.000000
mean,2024-07-06 05:30:00,93.523611,171.494444,128.668056,6061.275000,173.406944,1.493741,6232.769444,-44.738889,-0.116894,6249.768985,6268.248141,-25888.534722,137.869283,2.013512
min,2023-01-12 00:00:00,0.000000,7.000000,0.000000,1972.000000,0.000000,1.000000,2002.000000,-465.000000,-11.601513,2011.285714,2015.142857,-33908.000000,5.537749,0.000000
25%,2023-10-16 18:00:00,12.000000,36.000000,14.000000,2467.750000,19.750000,1.000000,2500.750000,-94.250000,-0.689345,2494.000000,2485.839286,-32173.750000,42.544987,0.862857
50%,2024-07-05 12:00:00,99.000000,193.000000,157.000000,6406.500000,181.000000,1.000000,6633.500000,-11.500000,0.123877,6621.642857,6605.035714,-31494.000000,111.513399,1.278049
75%,2025-03-25 06:00:00,147.250000,263.250000,199.250000,8010.250000,267.000000,1.000000,8249.500000,5.250000,0.858314,8270.714286,8271.607143,-20965.750000,192.422286,2.032520
max,2025-12-21 00:00:00,333.000000,531.000000,440.000000,11516.000000,505.000000,10.000000,11762.000000,206.000000,8.339628,11459.714286,11357.714286,-402.000000,817.305707,43.272727
std,NaN,72.646625,126.354965,97.322012,2833.070109,125.702841,1.001198,2918.458078,95.863856,1.895490,2907.833293,2894.893665,9665.680327,122.272489,3.062553


- Exporting the Cleaned Dataset

In [32]:
df.to_csv('Cleaned_HHS_Care_Dataset.csv', index=False)

- Understanading the Cleaned Dataset some more

In [ ]:
df.shape

(720, 19)

In [ ]:
df.isnull().sum()

Date                          0
CBP_Intake                    0
CBP_Custody                   0
HHS_Transfers                 0
HHS_Care                      0
HHS_Discharges                0
Days_Since_Previous_Report    1
Transfer_Exceeds_CBP          0
Discharge_Exceeds_HHS         0
Total_System_Load             0
Net_Intake                    0
Care_Load_Growth_Rate         1
Rolling_7D_Load               2
Rolling_14D_Load              6
Cumulative_Net_Intake         0
Positive_Net_Intake           0
Load_Volatility_7_Obs         2
Discharge_Offset_Ratio        3
Anomaly_Flag                  0
dtype: int64

In [ ]:
anomalies.value_counts()

Date        CBP_Intake  CBP_Custody  HHS_Transfers  HHS_Care  HHS_Discharges  Days_Since_Previous_Report  Transfer_Exceeds_CBP  Discharge_Exceeds_HHS
2023-01-24  47          42           47             7433      175             1.0                         True                  False                    1
2023-01-25  20          22           41             7538      180             1.0                         True                  False                    1
2023-02-02  15          13           23             7879      298             1.0                         True                  False                    1
2023-02-22  107         215          230            7978      232             1.0                         True                  False                    1
2023-02-23  101         162          178            7914      386             1.0                         True                  False                    1
2023-03-01  84          158          193            7930      307          

- Understanding Anomalies better

In [ ]:
print("Transfer > CBP custody:", df["Transfer_Exceeds_CBP"].sum())
print("Discharge > HHS care:", df["Discharge_Exceeds_HHS"].sum())
print("\nAnomaly percentage:", round(df["Transfer_Exceeds_CBP"].mean() * 100,2),"%")

Transfer > CBP custody: 86
Discharge > HHS care: 0

Anomaly percentage: 11.94 %


In [ ]:
df[
    df["Transfer_Exceeds_CBP"]
][
    [
        "Date",
        "CBP_Custody",
        "HHS_Transfers"
    ]
].tail(10)

,Date,CBP_Custody,HHS_Transfers
486,2025-01-08,80,82
493,2025-01-19,152,168
494,2025-01-20,118,136
495,2025-01-21,64,117
497,2025-01-23,75,97
502,2025-01-30,42,47
503,2025-02-02,22,41
508,2025-02-09,13,23
512,2025-02-13,10,23
521,2025-02-27,15,17


#### From the above output, we can now finally establish that:
- 720 reporting observations
- 86 / 720 = 11.94% have HHS_Transfers > CBP_Custody
- 0 have HHS_Discharges > HHS_Care
- Reporting is 'irregular', with gaps of several days between observations
- Keeping the 86 anomalies and flagging them, to prevent any false prediction
- The NaNs are simply logical and thus should not be imputed

#### The fact that anomalies still occur in January–February 2025 tells us they're not simply an early-data problem

"Although the project specification describes daily observations, the supplied dataset contains irregular reporting intervals. Therefore, missing calendar dates were treated as unobserved reporting periods rather than zero-care observations."ality discussion.

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date                        720 non-null    datetime64[us]
 1   CBP_Intake                  720 non-null    int64         
 2   CBP_Custody                 720 non-null    int64         
 3   HHS_Transfers               720 non-null    int64         
 4   HHS_Care                    720 non-null    int64         
 5   HHS_Discharges              720 non-null    int64         
 6   Days_Since_Previous_Report  719 non-null    float64       
 7   Transfer_Exceeds_CBP        720 non-null    bool          
 8   Discharge_Exceeds_HHS       720 non-null    bool          
 9   Total_System_Load           720 non-null    int64         
 10  Net_Intake                  720 non-null    int64         
 11  Care_Load_Growth_Rate       719 non-null    float64       
 12  Rolli